# 🛡️ DrugGuard — Text Model Training (DistilBERT)

This notebook fine-tunes **DistilBERT** for drug trafficking text detection.

**Requirements:**
- Google Colab with **GPU runtime** (Runtime → Change runtime type → T4 GPU)
- Upload `dataset.csv` to Colab session storage

**Outputs:**
- `text_model.onnx` — Lightweight model for CPU inference
- `text_model.pkl` + `vectorizer.pkl` — Fallback TF-IDF model
- `training_metrics.json` — Accuracy, F1, precision, recall

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers datasets torch scikit-learn pandas onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 39.1 MB/s eta 0:00:00


## Step 2: Upload Dataset

Upload your `dataset.csv` file using the file browser on the left (📁 icon), or run the cell below:

In [ ]:
import os

if not os.path.exists('dataset.csv'):
    from google.colab import files
    uploaded = files.upload()  # Upload dataset.csv
    print(f'Uploaded: {list(uploaded.keys())}')
else:
    print('dataset.csv already exists!')

dataset.csv already exists!


## Step 3: Load & Explore Dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('dataset.csv')
print(f'Dataset shape: {df.shape}')
print(f'\nLabel distribution:')
print(df['label'].value_counts())
print(f'\nBalance: {df["label"].mean()*100:.1f}% drug / {(1-df["label"].mean())*100:.1f}% safe')
print(f'\nSample rows:')
df.sample(5)

Dataset shape: (5500, 2)

Label distribution:
label
0    2750
1    2750
Name: count, dtype: int64

Balance: 50.0% drug / 50.0% safe

Sample rows:


,text,label
2484,Are you interested in reading this textbook ab...,0
4384,Can you pick up my ibuprofen refill from the C...,0
3852,Got a fresh batch of h*roin ready for delivery...,1
5055,"The weather is absolutely beautiful outside, l...",0
3884,"Cheap 0xy pills for sale, bulk discounts avail...",1


## Step 4: Text Normalization (Emoji + Algospeak)

In [ ]:
EMOJI_MAP = {
    '🍃': ' marijuana ', '🌿': ' marijuana ', '❄️': ' cocaine ',
    '💊': ' pills ', '🔌': ' plug ', '🍄': ' mushrooms ',
    '💉': ' heroin ', '📦': ' package ', '💰': ' cash ',
    '🔥': ' fire ', '⚡': ' fast ', '🧊': ' ice ',
    '🌬️': ' smoke ', '🎱': ' eightball ', '🏔️': ' mountain ',
    '🍬': ' candy ', '🧪': ' lab ', '🔬': ' tested ', '💯': ' hundred ',
}

LEETSPEAK_MAP = {
    'dr*gs': 'drugs', 'dr*g': 'drug',
    'd r u g s': 'drugs', 'd r u g': 'drug',
    'x*nax': 'xanax', 'x@nax': 'xanax', 'x@n@x': 'xanax',
    'c*caine': 'cocaine', 'c0ca1ne': 'cocaine',
    'h*roin': 'heroin', 'h.e.r.o.i.n': 'heroin',
    'f*ntanyl': 'fentanyl', 'f3nt@nyl': 'fentanyl',
    'm3th': 'meth', 'p1lls': 'pills', 'p3rc0c3t': 'percocet',
    'w33d': 'weed', 'mar1juana': 'marijuana',
    'k3tam1ne': 'ketamine', 'c@nnab!s': 'cannabis',
    'g@nja': 'ganja', '0xy': 'oxy',
}

def normalize_text(text: str) -> str:
    for emoji, rep in EMOJI_MAP.items():
        text = text.replace(emoji, rep)
    for pattern, rep in LEETSPEAK_MAP.items():
        text = text.replace(pattern, rep)
    return text

df['text'] = df['text'].astype(str).apply(normalize_text).str.lower()
print('Text normalization complete!')
df.head()

Text normalization complete!


,text,label
0,ensure you take your aspirin dose after meals.,0
1,"premium quality w e e d available, pgp verifie...",1
2,the company announced a significant increase i...,0
3,the company announced a significant increase i...,0
4,the government is implementing strict border c...,0


## Step 5: Train TF-IDF + Logistic Regression (Fallback Model)

In [ ]:
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score

X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(
    max_features=10000, ngram_range=(1, 2), min_df=2,
    token_pattern=r'(?u)\b\w+\b|[^\x00-\x7F]'
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

lr_model = LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced', random_state=42)
lr_model.fit(X_train_vec, y_train)

y_pred_lr = lr_model.predict(X_test_vec)
lr_acc = accuracy_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)

print(f'TF-IDF + LR Accuracy: {lr_acc*100:.2f}%')
print(f'TF-IDF + LR F1 Score: {lr_f1*100:.2f}%')
print(f'\n{classification_report(y_test, y_pred_lr, target_names=["Safe", "Drug"])}')

with open('text_model.pkl', 'wb') as f:
    pickle.dump(lr_model, f)
with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print('Saved: text_model.pkl, vectorizer.pkl')

TF-IDF + LR Accuracy: 100.00%
TF-IDF + LR F1 Score: 100.00%

              precision    recall  f1-score   support

        Safe       1.00      1.00      1.00       550
        Drug       1.00      1.00      1.00       550

    accuracy                           1.00      1100
   macro avg       1.00      1.00      1.00      1100
weighted avg       1.00      1.00      1.00      1100

Saved: text_model.pkl, vectorizer.pkl


## Step 6: Fine-tune DistilBERT (GPU Required)

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ No GPU detected! Enable GPU: Runtime → Change runtime type → T4 GPU')

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import Dataset
from sklearn.metrics import precision_score, recall_score

train_df_bert = df.iloc[X_train.index].reset_index(drop=True)
test_df_bert = df.iloc[X_test.index].reset_index(drop=True)

train_ds = Dataset.from_pandas(train_df_bert[['text', 'label']])
test_ds = Dataset.from_pandas(test_df_bert[['text', 'label']])

model_name = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

print(f'Tokenized: Train={len(train_ds)}, Test={len(test_ds)}')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/4400 [00:00<?, ? examples/s]

Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

Tokenized: Train=4400, Test=1100


In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

training_args = TrainingArguments(
    output_dir='./distilbert_output',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=50,
    report_to='none',
    fp16=torch.cuda.is_available(),
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds),
        'precision': precision_score(labels, preds),
        'recall': recall_score(labels, preds),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

print('Starting DistilBERT training... (takes ~10-15 min on T4 GPU)')
trainer.train()

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.001676,0.000923,1.000000,1.000000,1.000000,1.000000
2,0.000539,0.000359,1.000000,1.000000,1.000000,1.000000
3,0.000385,0.000275,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=825, training_loss=0.048488722452611634, metrics={'train_runtime': 83.3654, 'train_samples_per_second': 158.339, 'train_steps_per_second': 9.896, 'total_flos': 437142415564800.0, 'train_loss': 0.048488722452611634, 'epoch': 3.0})

In [ ]:
results = trainer.evaluate()
print('\n' + '='*50)
print(f'  DistilBERT Results:')
print(f'  Accuracy  : {results["eval_accuracy"]*100:.2f}%')
print(f'  F1 Score  : {results["eval_f1"]*100:.2f}%')
print(f'  Precision : {results["eval_precision"]*100:.2f}%')
print(f'  Recall    : {results["eval_recall"]*100:.2f}%')
print('='*50)

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.000385,0.000923,3,1.000000,1.000000,1.000000,1.000000



  DistilBERT Results:
  Accuracy  : 100.00%
  F1 Score  : 100.00%
  Precision : 100.00%
  Recall    : 100.00%


## Step 7: Export to ONNX (for CPU Inference)

In [ ]:
# Install onnxscript if not already installed
try:
    import onnxscript
except ImportError:
    !pip install onnxscript
    import onnxscript

# Save model folder
model.save_pretrained('./text_model_distilbert')
tokenizer.save_pretrained('./text_model_distilbert')

# Export to ONNX
model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dummy = tokenizer('sample text', return_tensors='pt', padding='max_length', truncation=True, max_length=128)
dummy = {k: v.to(device) for k, v in dummy.items()}

torch.onnx.export(
    model,
    (dummy['input_ids'], dummy['attention_mask']),
    'text_model.onnx',
    input_names=['input_ids', 'attention_mask'],
    output_names=['logits'],
    dynamic_axes={
        'input_ids': {0: 'batch_size', 1: 'seq_len'},
        'attention_mask': {0: 'batch_size', 1: 'seq_len'},
        'logits': {0: 'batch_size'},
    },
    opset_version=14,
)
print('Exported: text_model.onnx')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 20.7 MB/s eta 0:00:00


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/tmp/ipykernel_1075/3827478438.py:18: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0804 16:26:21.540000 1075 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `DistilBertForSequenceClassification([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DistilBertForSequenceClassification([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /project/onnx/version_converter/adapters/no_previous_version.h:24: adapt: Assertion `false` fa

[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/exporter/_onnx_program.py:487: UserWarning: # The axis name: seq_len will not be used, since it shares the same shape constraints with another axis: seq_len.
  rename_mapping = _dynamic_shapes.create_rename_mapping(


Exported: text_model.onnx


## Step 8: Download Model Files

In [ ]:
import json
from google.colab import files

metrics = {
    'distilbert': {
        'accuracy': round(results['eval_accuracy'], 4),
        'f1_score': round(results['eval_f1'], 4),
    },
    'tfidf_lr': {
        'accuracy': round(lr_acc, 4),
        'f1_score': round(lr_f1, 4),
    }
}
with open('training_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Downloading files to local machine...')
files.download('text_model.onnx')
files.download('text_model.pkl')
files.download('vectorizer.pkl')
files.download('training_metrics.json')
print('Done!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done!
